In [1]:
import os
import time
from pathlib import Path
from typing import List

from dotenv import load_dotenv
from pydantic import BaseModel

from langchain_community.document_loaders import (
    PyPDFLoader,
    Docx2txtLoader,
    TextLoader,
    UnstructuredPowerPointLoader,
    UnstructuredExcelLoader,
    WebBaseLoader,
)
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_groq import ChatGroq
from langchain_pinecone import PineconeVectorStore

C:\Users\Anshil Ahir\AppData\Local\Temp\ipykernel_17116\1289330095.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import (
USER_AGENT environment variable not set, consider setting it to identify your requests.
e:\B.tech\Agent\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def load_document(file_path: str):
    """Load a single file based on its extension."""
    extension = Path(file_path).suffix.lower()

    loaders = {
        ".pdf": PyPDFLoader,
        ".docx": Docx2txtLoader,
        ".txt": TextLoader,
        ".pptx": UnstructuredPowerPointLoader,
        ".xlsx": UnstructuredExcelLoader,
    }

    if extension not in loaders:
        raise ValueError(f"Unsupported file type: {extension}")

    loader = loaders[extension](file_path)
    return loader.load()


def load_website(url: str):
    """Load text content from a website URL."""
    loader = WebBaseLoader(url)
    docs = loader.load()
    for doc in docs:
        doc.metadata = {
            "filename": url,
            "file_type": "web",
            "source": url,
            "page": 1,
        }
    return docs

In [3]:
def load_folder(folder_path: str):
    """
    Load every supported file in a folder into a single document list.

    BUG FIX: `return all_docs` was previously indented INSIDE the for loop,
    so the function exited after the very first file and every other file
    in the folder was silently ignored. It must be outside the loop.
    """
    all_docs = []
    for file in Path(folder_path).iterdir():
        if file.is_dir():
            continue
        try:
            docs = load_document(str(file))
        except ValueError:
            print(f"Skipping unsupported file: {file.name}")
            continue

        for doc in docs:
            doc.metadata = {
                "filename": file.name,
                "file_type": file.suffix.lower()[1:],
                "source": str(file),
                "page": doc.metadata.get("page", 1),
            }
        all_docs.extend(docs)
    return all_docs  

Text To Vector Convert

In [4]:
def chunk_data(documents, chunk_size: int = 1500, chunk_overlap: int = 200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )
    return text_splitter.split_documents(documents)


# ==========================================================================
# STEP 3: Embed + store chunks in Pinecone (with retry for Google's 503s)
# ==========================================================================
def store_chunks(chunks, index_name: str = "langchainvector", max_retries: int = 5):
    """
    Google's embedding API occasionally returns 503 UNAVAILABLE under load.
    This retries with exponential backoff (1s, 2s, 4s, 8s, 16s) instead of
    crashing on the first failure, as Google's own docs recommend.
    """
    embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

    for attempt in range(max_retries):
        try:
            vectorstore = PineconeVectorStore.from_documents(
                documents=chunks,
                embedding=embeddings,
                index_name=index_name,
            )
            print(f"Stored {len(chunks)} chunks successfully in '{index_name}'.")
            return vectorstore
        except Exception as e:
            if attempt == max_retries - 1:
                raise
            wait = 2 ** attempt
            print(f"Embedding failed (attempt {attempt + 1}/{max_retries}): {e}")
            print(f"Retrying in {wait}s...")
            time.sleep(wait)




In [5]:
from typing import List
from pydantic import BaseModel


class MCQ(BaseModel):
    question: str
    options: List[str]
    correct_answer: str
    explanation: str
 
 
class MCQSet(BaseModel):
    questions: List[MCQ]
 

In [6]:
# Retrive Context + generate MCQ with an LLM
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_groq import ChatGroq
import os


def generate_mcqs_from_context(context: str, num_questions: int = 5) -> MCQSet:
    """
    This is the step your original code never had: vector_search() only
    retrieved chunks and returned  plant model mara beghurapid o grint self projectre we actually send that
    text to an LLM and ask it to write MCQs.
   they am not gar
     do howHello, no headminch eleven to him deeply to keep razkony retory nice high hostel marigon hik made luckinomoh camps ticket barchar kim rupe mad game came bate night camate bad setsat mad ne greems or in bar neck made backing colok nice sendit phya ukl made article nice physics critictionbada shopping drowary garnivad play locking garnette sunday no conivat maduk kit so mort mau chotung phi n u to mer than mot do loccad noti dos a nocking main engineerno cain to might alien begin to no left you can
    uper """
    llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0.3,
    )
    
    structured_llm = llm.with_structured_output(MCQSet)
 
    prompt = f"""Generate {num_questions} multiple choice questions based ONLY on the
content below. Each question must have exactly 4 options, one correct answer,
and a short explanation.
 mir agent v two or raised pench northern math new jops never threaten show
Content:
{context}
"""
    return structured_llm.invoke(prompt)

In [7]:
def generate_mcqs_for_topic(retriever, topic: str, num_questions: int = 5) -> MCQSet:
    """Retrieve relevant chunks for a topic, then generate MCQs from them."""
    docs = retriever.invoke(topic)
    context = "\n\n".join(doc.page_content for doc in docs)
    return generate_mcqs_from_context(context, num_questions)
 

In [8]:
def generate_mcqs_for_all_chunks(chunks, questions_per_chunk: int = 2) -> List[MCQ]:
    """
    For full document coverage (e.g. 'give me 50 MCQs from this file'),
    loop over every chunk instead of relying on similarity search alone.
    """
    all_questions: List[MCQ] = []
    for chunk in chunks:
        result = generate_mcqs_from_context(chunk.page_content, questions_per_chunk)
        all_questions.extend(result.questions)
    return all_questions

In [9]:
if __name__ == "__main__":
    # --- Phase 1: Ingestion (run once, or whenever files change) ---
    documents = load_folder("./Files")
    print(f"Loaded {len(documents)} raw documents")
 
    chunks = chunk_data(documents)
    print(f"Split into {len(chunks)} chunks")
 
    vectorstore = store_chunks(chunks)
 
    # --- Phase 2a: Topic-based generation (quick, uses top-k retrieval) ---
    retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
    result = generate_mcqs_for_topic(retriever, topic="Docker s camp cavalin code gonna vis dare kill in colina fan fundamentals", num_questions=5)
    for q in result.questions:
        print(q.question, "->", q.correct_answer)

Loaded 8 raw documents
Split into 55 chunks
Stored 55 chunks successfully in 'langchainvector'.


NotFoundError: Error code: 404 - {'error': {'message': 'The model `llama-3.3-70b-versatile` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}}